# Conquer Online — Original Reference Image Pack Builder

A standalone Kaggle notebook that collects publicly accessible reference images from official Conquer Online / Conquer Online Mobile guide pages, validates them, removes duplicate files, records source URLs, and exports one ZIP archive for the separate SDXL redesign + Hunyuan3D notebook.

**Output:** `/kaggle/working/conquer_original_reference_images.zip` plus an extracted folder and CSV manifest.

**Important:** This collects images embedded in the listed public web pages. It does not extract proprietary game-client archives, sprites, meshes, or hidden game files. Page layouts and remote access can change; the report records pages that could not be read.

## How to use
1. Open this notebook in Kaggle.
2. Enable Internet in Notebook options.
3. A GPU is not required; CPU is sufficient.
4. Run All.
5. Download `conquer_original_reference_images.zip` from the Output panel and attach it as a Kaggle Dataset/input to the separate Hunyuan3D redesign notebook.


In [ ]:
!pip -q install beautifulsoup4 pillow requests


In [ ]:
from pathlib import Path
from urllib.parse import urljoin, urlparse, unquote
from bs4 import BeautifulSoup
from PIL import Image
from zipfile import ZipFile, ZIP_DEFLATED
import requests, hashlib, csv, json, re, time, os

WORK = Path('/kaggle/working')
PACK_DIR = WORK / 'conquer_original_reference_images'
PACK_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = WORK / 'conquer_original_reference_images.zip'
MANIFEST_PATH = PACK_DIR / 'sources_manifest.csv'
REPORT_PATH = PACK_DIR / 'collection_report.json'

# Official public guide/gallery pages used as image sources.
SOURCE_PAGES = {
    'characters': [
        'https://comobile.99.com/guide/info/classes/2023-01-31/classes_-_classes_introduction.shtml',
    ],
    'maps': [
        'https://comobile.99.com/guide/info/maps/2023-02-17/guide_-_maps.shtml',
    ],
    'mounts': [
        'https://comobile.99.com/guide/info/classes/2023-02-14/status_-_mount_system.shtml',
    ],
    'equipment': [
        'https://comobile.99.com/guide/info/equipment/2023-02-15/equipment_-_equipment_type.shtml',
    ],
    'monsters': [
        'https://co.99.com/downloads/wallpapers_monster.shtml',
    ],
}

ALLOWED_HOSTS = {'comobile.99.com', 'co.99.com', 'hw.99.com', 'images.99.com'}
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
MAX_IMAGES_PER_CATEGORY = 500
MIN_IMAGE_BYTES = 5_000
MAX_IMAGE_BYTES = 30_000_000
REQUEST_DELAY_SECONDS = 0.15

session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36'})
manifest_rows, page_report, seen_hashes, seen_urls = [], [], set(), set()

def allowed_url(url):
    p = urlparse(url)
    return p.scheme in ('http', 'https') and (p.hostname or '').lower() in ALLOWED_HOSTS

def fetch_html(url):
    response = session.get(url, timeout=35)
    response.raise_for_status()
    response.encoding = response.apparent_encoding or response.encoding
    return response.text

def candidate_image_urls(html, page_url):
    soup = BeautifulSoup(html, 'html.parser')
    found = []
    attrs = ('src', 'data-src', 'data-original', 'data-lazy-src', 'href')
    for tag in soup.find_all(['img', 'a', 'source']):
        values = [tag.get(a) for a in attrs if tag.get(a)]
        srcset = tag.get('srcset') or tag.get('data-srcset')
        if srcset:
            values.extend(part.strip().split()[0] for part in srcset.split(',') if part.strip())
        for raw in values:
            raw = raw.strip().replace('&amp;', '&')
            full = urljoin(page_url, raw)
            if allowed_url(full) and Path(urlparse(full).path).suffix.lower() in IMAGE_EXTS:
                found.append(full)
    # Include CSS background-image URLs used by some older guide layouts.
    for raw in re.findall(r'url\([\"\']?([^\)\"\']+)', html, flags=re.I):
        full = urljoin(page_url, raw.strip())
        if allowed_url(full) and Path(urlparse(full).path).suffix.lower() in IMAGE_EXTS:
            found.append(full)
    return list(dict.fromkeys(found))

def discover_related_class_pages(html, page_url):
    soup = BeautifulSoup(html, 'html.parser')
    pages = []
    for a in soup.find_all('a', href=True):
        full = urljoin(page_url, a['href'])
        path = urlparse(full).path.lower()
        if allowed_url(full) and '/guide/info/classes/' in path and full not in pages:
            pages.append(full)
    return pages[:40]

def safe_filename(url, category, index):
    base = unquote(Path(urlparse(url).path).name) or f'image_{index}.jpg'
    base = re.sub(r'[^A-Za-z0-9._-]+', '_', base)[:100]
    if Path(base).suffix.lower() not in IMAGE_EXTS:
        base += '.jpg'
    return f'{category}_{index:04d}_{base}'

def collect_category(category, pages):
    category_dir = PACK_DIR / category
    category_dir.mkdir(parents=True, exist_ok=True)
    category_urls = []
    for page_url in pages:
        try:
            html = fetch_html(page_url)
            found = candidate_image_urls(html, page_url)
            category_urls.extend(found)
            page_report.append({'category': category, 'url': page_url, 'status': 'read', 'image_candidates': len(found)})
            if category == 'characters':
                for related in discover_related_class_pages(html, page_url):
                    try:
                        related_html = fetch_html(related)
                        related_images = candidate_image_urls(related_html, related)
                        category_urls.extend(related_images)
                        page_report.append({'category': category, 'url': related, 'status': 'read_related', 'image_candidates': len(related_images)})
                    except Exception as exc:
                        page_report.append({'category': category, 'url': related, 'status': 'failed', 'error': str(exc)[:300]})
                    time.sleep(REQUEST_DELAY_SECONDS)
        except Exception as exc:
            page_report.append({'category': category, 'url': page_url, 'status': 'failed', 'error': str(exc)[:300]})
    saved = 0
    for url in dict.fromkeys(category_urls):
        if url in seen_urls or saved >= MAX_IMAGES_PER_CATEGORY:
            continue
        seen_urls.add(url)
        try:
            response = session.get(url, timeout=45)
            response.raise_for_status()
            data = response.content
            if not MIN_IMAGE_BYTES <= len(data) <= MAX_IMAGE_BYTES:
                continue
            digest = hashlib.sha256(data).hexdigest()
            if digest in seen_hashes:
                continue
            destination = category_dir / safe_filename(url, category, saved + 1)
            destination.write_bytes(data)
            try:
                with Image.open(destination) as im:
                    im.verify()
                with Image.open(destination) as im:
                    width, height, fmt = im.width, im.height, im.format
            except Exception:
                destination.unlink(missing_ok=True)
                continue
            seen_hashes.add(digest)
            manifest_rows.append({'category': category, 'file': str(destination.relative_to(PACK_DIR)), 'bytes': len(data), 'width': width, 'height': height, 'format': fmt, 'sha256': digest, 'source_url': url})
            saved += 1
            time.sleep(REQUEST_DELAY_SECONDS)
        except Exception as exc:
            print(f'[SKIP] {url} :: {str(exc)[:180]}')
    print(f'{category}: saved {saved} valid image(s)')

for category, pages in SOURCE_PAGES.items():
    collect_category(category, pages)

with MANIFEST_PATH.open('w', newline='', encoding='utf-8-sig') as f:
    fields = ['category', 'file', 'bytes', 'width', 'height', 'format', 'sha256', 'source_url']
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(manifest_rows)

summary = {
    'total_valid_images': len(manifest_rows),
    'categories': {name: sum(1 for row in manifest_rows if row['category'] == name) for name in SOURCE_PAGES},
    'total_image_bytes': sum(row['bytes'] for row in manifest_rows),
    'pages': page_report,
    'source_pages': SOURCE_PAGES,
    'note': 'Only publicly accessible images embedded in listed guide/gallery pages are collected; this is not a game-client asset extractor.'
}
REPORT_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

with ZipFile(ZIP_PATH, 'w', ZIP_DEFLATED, compresslevel=6) as archive:
    for file_path in PACK_DIR.rglob('*'):
        if file_path.is_file():
            archive.write(file_path, file_path.relative_to(PACK_DIR.parent))

print('\n=== COLLECTION SUMMARY ===')
print('Verified images:', len(manifest_rows))
for category, count in summary['categories'].items():
    print(f'  {category}: {count}')
print(f"ZIP created: {ZIP_PATH} ({ZIP_PATH.stat().st_size / (1024*1024):.2f} MB)")
print('Manifest:', MANIFEST_PATH)
print('Report:', REPORT_PATH)
if not manifest_rows:
    raise RuntimeError('No valid images were collected. Check Kaggle Internet access and inspect collection_report.json for source-page failures.')


## Attach the ZIP to the redesign notebook

After the ZIP is created, download it from Kaggle Output. Create a Kaggle Dataset from the ZIP (or upload the ZIP as a Dataset file), then attach that Dataset to the separate `hunyuan3d_2_1_kaggle_dual_t4_clash_of_legends.ipynb` notebook using **Add Input → Your Datasets**.

The redesign notebook can then read the image files under `/kaggle/input/.../conquer_original_reference_images/`. Keep this collector notebook separate; it only gathers and packages references.
